# <span style="font-width:bold; font-size: 3rem; color:#1EB182;"> **Air Quality** </span><span style="font-width:bold; font-size: 3rem; color:#333;">- Part 04: Multi-Day Batch Inference</span>

## 🗒️ This notebook is divided into the following sections:

1. Download model and batch inference data
2. Make multi-day predictions, generate PNG for forecast
3. Store predictions in a monitoring feature group and generate PNG for hindcast

## <span style='color:#ff5f27'> 📝 Imports

In [ ]:
import sys
from pathlib import Path

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/featurestorebook/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml


if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    # Strip ~/notebooks/ccfraud from PYTHON_PATH if notebook started in one of these subdirectories
    if root_dir.parts[-1:] == ('airquality',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    sys.path.append(root_dir)
print(f"Added the following directory to the PYTHONPATH: {root_dir}")
    
# Read the API keys and configuration variables from the file <root_dir>/.env
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

In [ ]:
import datetime
import pandas as pd
from xgboost import XGBRegressor
import hopsworks
import json
from mlfs.airquality import util
import os

## <span style='color:#ff5f27'> 🔧 Configuration: Set Forecast Days</span>

In [ ]:
# Configuration: Set number of days to forecast (can be changed to 3, 7, 14, etc.)
# You can also set this in your .env file as FORECAST_DAYS=7
FORECAST_DAYS = int(os.getenv('FORECAST_DAYS', 7))
print(f"Will forecast for the next {FORECAST_DAYS} days")

In [ ]:
today = datetime.datetime.now() - datetime.timedelta(0)
# Generate list of dates for the next N days
forecast_dates = [today + datetime.timedelta(days=i) for i in range(1, FORECAST_DAYS + 1)]
print(f"Forecast dates: {[d.strftime('%Y-%m-%d') for d in forecast_dates]}")

## <span style="color:#ff5f27;"> 📡 Connect to Hopsworks Feature Store </span>

In [ ]:
project = hopsworks.login(engine="python")
fs = project.get_feature_store() 

secrets = hopsworks.get_secrets_api()
location_str = secrets.get_secret("SENSOR_LOCATION_JSON").value
location = json.loads(location_str)
country=location['country']
city=location['city']
street=location['street']

## <span style="color:#ff5f27;">🪝 Download the model from Model Registry</span>

In [ ]:
mr = project.get_model_registry()

retrieved_model = mr.get_model(
    name="air_quality_xgboost_model",
    version=1,
)

fv = retrieved_model.get_feature_view()

# Download the saved model artifacts to a local directory
saved_model_dir = retrieved_model.download()

In [ ]:
# Loading the XGBoost regressor model and label encoder from the saved model directory
retrieved_xgboost_model = XGBRegressor()

retrieved_xgboost_model.load_model(saved_model_dir + "/model.json")

# Displaying the retrieved XGBoost regressor model
retrieved_xgboost_model

## <span style="color:#ff5f27;">✨ Get Weather Forecast Features for Multiple Days</span>



In [ ]:
weather_fg = fs.get_feature_group(
    name='weather',
    version=1,
)

# Get all weather data from today to the last forecast date
last_forecast_date = forecast_dates[-1]
batch_data = weather_fg.filter(weather_fg.date >= today).filter(weather_fg.date <= last_forecast_date).read()
batch_data

### <span style="color:#ff5f27;">🤖 Making Multi-Day Predictions</span>

In [ ]:
# Make predictions for all retrieved data
batch_data['predicted_pm25'] = retrieved_xgboost_model.predict(
    batch_data[['temperature_2m_mean', 'precipitation_sum', 'wind_speed_10m_max', 'wind_direction_10m_dominant']])
batch_data

In [ ]:
batch_data.info()

### <span style="color:#ff5f27;">🤖 Saving the predictions (for monitoring) to a Feature Group</span>

In [ ]:
batch_data['street'] = street
batch_data['city'] = city
batch_data['country'] = country
# Fill in the number of days before the date on which you made the forecast (base_date)
batch_data['days_before_forecast_day'] = range(1, len(batch_data)+1)
batch_data = batch_data.sort_values(by=['date'])
batch_data

In [ ]:
batch_data.info()

### Create Multi-Day Forecast Graph
Draw a graph of the predictions with dates as a PNG and save it to the github repo
Show it on github pages

In [ ]:
pred_file_path = f"{root_dir}/docs/air-quality/assets/img/pm25_forecast_{FORECAST_DAYS}day.png"
plt = util.plot_air_quality_forecast(city, street, batch_data, pred_file_path)

plt.show()

In [ ]:
# Get or create feature group
monitor_fg = fs.get_or_create_feature_group(
    name='aq_predictions',
    description='Air Quality prediction monitoring',
    version=1,
    primary_key=['city','street','date','days_before_forecast_day'],
    event_time="date"
)

In [ ]:
monitor_fg.insert(batch_data, wait=True)

### <span style="color:#ff5f27;">📊 Compare Historical Data with Predictions</span>

In [ ]:
# Get actual data from the past 7 days for comparison
air_quality_fg = fs.get_feature_group(name='air_quality', version=1)
historical_start = today - datetime.timedelta(days=7)
historical_data = air_quality_fg.filter(air_quality_fg.date >= historical_start).filter(air_quality_fg.date <= today).read()

print(f"Historical data from {historical_start.strftime('%Y-%m-%d')} to {today.strftime('%Y-%m-%d')}")
historical_data

In [ ]:
# Create combined chart with historical and forecast data
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

# Plot historical data
if not historical_data.empty:
    plt.plot(historical_data['date'], historical_data['pm25'], 
             label='Historical PM2.5', marker='o', color='blue', linewidth=2)

# Plot forecast data
plt.plot(batch_data['date'], batch_data['predicted_pm25'], 
         label=f'{FORECAST_DAYS}-Day Forecast', marker='s', color='red', 
         linestyle='--', linewidth=2)

# Add today's dividing line
plt.axvline(x=today, color='green', linestyle=':', linewidth=2, label='Today')

plt.xlabel('Date')
plt.ylabel('PM2.5 Level')
plt.title(f'{city.title()} - {street.title()}\n{FORECAST_DAYS}-Day Air Quality Forecast with Historical Data')
plt.legend()
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save the chart
combined_file_path = f"{root_dir}/docs/air-quality/assets/img/pm25_combined_{FORECAST_DAYS}day.png"
plt.savefig(combined_file_path)
plt.show()

print(f"Combined chart saved to: {combined_file_path}")

### Upload the prediction dashboards (png files) to Hopsworks


In [ ]:
dataset_api = project.get_dataset_api()
str_today = today.strftime("%Y-%m-%d")
if dataset_api.exists("Resources/airquality") == False:
    dataset_api.mkdir("Resources/airquality")
dataset_api.upload(pred_file_path, f"Resources/airquality/{city}_{street}_{str_today}", overwrite=True)
dataset_api.upload(combined_file_path, f"Resources/airquality/{city}_{street}_{str_today}", overwrite=True)

proj_url = project.get_url()
print(f"See images in Hopsworks here: {proj_url}/settings/fb/path/Resources/airquality")

## <span style='color:#ff5f27'> 📈 Summary Statistics</span>

In [ ]:
# Display forecast statistics
print(f"\n=== {FORECAST_DAYS}-Day Forecast Summary ===")
print(f"Average predicted PM2.5: {batch_data['predicted_pm25'].mean():.2f}")
print(f"Maximum predicted PM2.5: {batch_data['predicted_pm25'].max():.2f}")
print(f"Minimum predicted PM2.5: {batch_data['predicted_pm25'].min():.2f}")
print(f"\nForecast period: {forecast_dates[0].strftime('%Y-%m-%d')} to {forecast_dates[-1].strftime('%Y-%m-%d')}")

---